# Exponential integrators in NGSolve — Part 3: non-symmetric operators and time-dependent sources

Parts 1 and 2 dealt with the heat equation: $A = M^{-1}K$ was self-adjoint in the $M$-inner product and the source term was constant. This notebook removes both restrictions.

1. **Convection–diffusion.** With a convective term the stiffness matrix is non-symmetric. The Lanczos three-term recurrence becomes the **Arnoldi** process with a full Hessenberg matrix, and the eigenvalues of $A$ move into the complex plane. Shift-and-invert still works and still converges independently of the mesh; the factorisation is now an LU factorisation (`umfpack`).
2. **Time-dependent sources** $f(t)$. The exact solution now involves an integral over the source,
$$
u(t_{n+1}) = e^{-\Delta t A}u(t_n) + \int_0^{\Delta t} e^{-(\Delta t - s)A}\, M^{-1} f(t_n + s)\, ds ,
$$
and approximating $f$ on $[t_n, t_{n+1}]$ by a polynomial yields the **exponential Runge–Kutta methods** of Hochbruck and Ostermann. Freezing $f(t_n)$ gives exponential Euler (order 1); linear interpolation gives the exponential trapezoidal rule (order 2) and requires $\varphi_2$ in addition to $\varphi_1$.

We reuse the model of tutorial 3.1: the square $(-1,1)^2$ with the rotating wind $b(x,y) = (2y(1-x^2), -2x(1-y^2))$ and diffusion $\varepsilon = 0.01$.

In [ ]:
import sys; sys.path.insert(0, "..")
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import numpy as np, scipy.linalg as sla, scipy.sparse as sp
import matplotlib.pyplot as plt
from expint_tools import phi, krylov_funm, KrylovExp, DenseReference
ngsglobals.msg_level = 0

In [ ]:
shape = Rectangle(2, 2).Face().Move((-1, -1, 0))
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.2))
fes = H1(mesh, order=3, dirichlet=".*")
u, v = fes.TnT()
b = CoefficientFunction((2*y*(1-x*x), -2*x*(1-y*y)))
eps = 0.01
a = BilinearForm(eps*grad(u)*grad(v)*dx + b*grad(u)*v*dx, symmetric=False).Assemble()   # non-symmetric K
m = BilinearForm(u*v*dx, symmetric=False).Assemble()
K, M = a.mat, m.mat
freedofs = fes.FreeDofs(); P = Projector(freedofs, True)
invM = M.Inverse(freedofs, inverse="sparsecholesky")

gfu = GridFunction(fes)
gfu.Set(exp(-20*((x+0.5)**2 + y**2)))
gfu.vec.data = P*gfu.vec
u0vec = gfu.vec.CreateVector(); u0vec.data = gfu.vec
print("ndof =", fes.ndof)
Draw(b, mesh, "wind", vectors={"grid_size": 24});
Draw(gfu, mesh, "u0");

## Dense reference

On this coarse mesh we can form $A = M^{-1}K$ densely and use `scipy.linalg.expm`. (The eigendecomposition-based `DenseReference` of the previous parts assumes a symmetric $K$; here we only use it for the export of the matrices and the $M$-norm.)

In [ ]:
ref = DenseReference(M, M, freedofs)         # only for idx, Md, to_np, norm
Kd = ref._dense(K); Md = ref.Md
Ad = np.linalg.solve(Md, Kd)                 # dense A = M^{-1} K on the free dofs
u0np = ref.to_np(u0vec)
def relerr(vec, refnp): return ref.norm(ref.to_np(vec) - refnp) / ref.norm(refnp)

ev = sla.eigvals(Ad)
print(f"eigenvalues of A:  Re in [{ev.real.min():.2f}, {ev.real.max():.3e}],  |Im| up to {abs(ev.imag).max():.2f}")

## Arnoldi instead of Lanczos

Everything in `krylov_funm` stays the same — `AppendOrthogonalize` orthogonalises against *all* previous basis vectors anyway — except that the projected matrix $H_m = V_m^T M A V_m$ is now upper Hessenberg instead of tridiagonal, and $f(H_m)e_1$ is evaluated with the (complex) eigendecomposition of $H_m$ or, if that is ill-conditioned, with the Schur–Parlett algorithm of `scipy.linalg.funm`. This is what `symmetric=False` selects.

Shift-and-invert works exactly as before with $B = (M + \gamma K)^{-1}M$ and the eigenvalue map $\lambda = (1/\theta - 1)/\gamma$, now for complex $\theta$; the factorisation of the non-symmetric $M + \gamma K$ uses `umfpack`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for dt, axis in zip([0.01, 0.1], ax):
    ex = sla.expm(-dt*Ad) @ u0np
    for poly, ms, lab in [(True, range(5, 121, 5), "polynomial Arnoldi"), (False, range(2, 31, 2), r"shift-and-invert Arnoldi, $\gamma = \Delta t/10$")]:
        E = KrylovExp(M, K, dt, freedofs, poly=poly, symmetric=False, tol=0)
        errs = []
        for mm in ms:
            E.maxm = mm; errs.append(relerr((E*u0vec).Evaluate(), ex))
        axis.semilogy(list(ms), errs, "o-", ms=3, label=lab)
    axis.set_title(rf"convection–diffusion, $\Delta t$ = {dt}"); axis.set_xlabel("m"); axis.set_ylabel("rel. error"); axis.grid(); axis.legend(); axis.set_ylim(1e-14, 10)
plt.tight_layout(); plt.show()

### Remark: spectrum versus field of values

For a non-normal operator the convergence of Krylov methods is not governed by the spectrum alone but by the **field of values** (numerical range) in the relevant inner product,
$W_M(A) = \{ (Au,u)_M / (u,u)_M \}$ — Crouzeix's theorem bounds $\|f(A)\|$ by $2\max_{W(A)}|f|$. Below the spectrum of $A$ and the boundary of its field of values (computed with Johnson's algorithm from the extreme eigenvectors of the Hermitian parts of $e^{-i\vartheta}\tilde A$, $\tilde A = R^{-T}KR^{-1}$, $M = R^TR$). The imaginary parts come from the convection; they are what forces the complex arithmetic in the small matrix.

In [ ]:
R = np.linalg.cholesky(Md).T                            # Md = R^T R
Rinv = np.linalg.inv(R)
At = Rinv.T @ Kd @ Rinv                                 # Euclidean-symmetric representative of A
fov = []
for th in np.linspace(0, 2*np.pi, 40, endpoint=False):
    Hth = 0.5*(np.exp(-1j*th)*At + np.exp(1j*th)*At.T)
    w = sla.eigh(Hth)[1][:, -1]
    fov.append((w.conj() @ At @ w) / (w.conj() @ w))
fov = np.array(fov)
plt.figure(figsize=(6, 3.5)); plt.plot(ev.real, ev.imag, ".", ms=3, label="eigenvalues of A")
plt.plot(fov.real, fov.imag, "r-", label="boundary of $W_M(A)$"); plt.xscale("symlog", linthresh=10)
plt.xlabel("Re"); plt.ylabel("Im"); plt.legend(); plt.grid(); plt.show()

## Time-dependent sources: exponential Runge–Kutta methods

With $g(t) := M^{-1}f(t)$ the variation-of-constants formula reads

$$
u(t_{n+1}) = e^{-\Delta tA}u(t_n) + \Delta t\int_0^1 e^{-(1-\tau)\Delta tA}\, g(t_n + \tau\Delta t)\, d\tau .
$$

* $g(t_n + \tau\Delta t) \approx g(t_n)$ gives the **exponential Euler** method, $u^{n+1} = e^{-\Delta tA}u^n + \Delta t\,\varphi_1(-\Delta tA)\,g(t_n)$, of order 1.
* Linear interpolation $g(t_n+\tau\Delta t) \approx (1-\tau)g(t_n) + \tau g(t_{n+1})$ gives the **exponential trapezoidal rule**
$$
u^{n+1} = e^{-\Delta tA}u^n + \Delta t\Big[\big(\varphi_1 - \varphi_2\big)(-\Delta tA)\, g(t_n) + \varphi_2(-\Delta tA)\, g(t_{n+1})\Big],
\qquad \varphi_2(z) = \frac{e^z - 1 - z}{z^2},
$$
of order 2 — since $\int_0^1 e^{(1-\tau)z}\tau\,d\tau = \varphi_2(z)$. It is the simplest member of the exponential Runge–Kutta family (Hochbruck–Ostermann 2005); higher orders use more $\varphi_k$ and internal stages.

Both are compared with implicit Euler and Crank–Nicolson for $f(t) = \sin(2\pi t)\, f_1(x)$. For this source the exact solution of the semi-discrete problem is available in closed form:
$\int_0^T e^{-(T-s)A}\sin(\omega s)\,ds = \mathrm{Im}\big[(A + i\omega I)^{-1}(e^{i\omega T}I - e^{-TA})\big]$.

All $\varphi_k$-terms are obtained from the *same* `KrylovExp` object through `E.Apply(w, func)` — different scalar functions, same Krylov process, same factorisation.

In [ ]:
tpar = Parameter(0.0)
omega = 2*pi
f1 = exp(-20*((x-0.5)**2 + (y-0.2)**2))
f = LinearForm(fes); f += sin(omega*tpar)*f1*v*dx
def fvec(t):
    tpar.Set(t); f.Assemble(); return f.vec

Tend = 0.5
f1np = ref.to_np(fvec(0.25 + 0*0.0)) / np.sin(omega*0.25)              # M f_1 on the free dofs
g1np = np.linalg.solve(Md, f1np)
ET = sla.expm(-Tend*Ad)
exact_T = ET @ u0np + np.imag(np.linalg.solve(Ad + 1j*omega*np.eye(len(u0np)), np.exp(1j*omega*Tend)*g1np - ET @ g1np))

In [ ]:
def theta_scheme(theta, dt):
    mstar = M.CreateMatrix(); mstar.AsVector().data = M.AsVector() + theta*dt*K.AsVector()
    inv = mstar.Inverse(freedofs, inverse="umfpack")
    w = u0vec.CreateVector(); w.data = u0vec; n = int(round(Tend/dt))
    for k in range(n):
        rhs = (M*w - (1-theta)*dt*(K*w) + (1-theta)*dt*fvec(k*dt)).Evaluate()      # f at t_n ...
        rhs.data += theta*dt*fvec((k+1)*dt)                                        # ... and t_{n+1}
        w.data = inv*rhs
    return w

def exp_euler(dt, tol=1e-9):
    E = KrylovExp(M, K, dt, freedofs, symmetric=False, tol=tol)
    w = u0vec.CreateVector(); w.data = u0vec; n = int(round(Tend/dt))
    for k in range(n):
        w.data = E*w + dt*E.Phi(1, invM*fvec(k*dt))
    return w

def exp_trapezoidal(dt, tol=1e-9):
    E = KrylovExp(M, K, dt, freedofs, symmetric=False, tol=tol)
    w = u0vec.CreateVector(); w.data = u0vec; n = int(round(Tend/dt))
    gnext = (invM*fvec(0.0)).Evaluate()
    for k in range(n):
        gnow, gnext = gnext, (invM*fvec((k+1)*dt)).Evaluate()
        w.data = E*w + dt*E.Apply(gnow, lambda lam: phi(1, -dt*lam) - phi(2, -dt*lam)) + dt*E.Phi(2, gnext)
    return w

dts = [0.05, 0.025, 0.0125, 0.00625]
methods = {"implicit Euler": lambda dt: theta_scheme(1.0, dt), "Crank-Nicolson": lambda dt: theta_scheme(0.5, dt),
           "exponential Euler": exp_euler, "exponential trapezoidal": exp_trapezoidal}
err = {k: [relerr(fun(dt), exact_T) for dt in dts] for k, fun in methods.items()}
print(f"{'dt':>8s} " + " ".join(f"{k:>24s}" for k in err))
for i, dt in enumerate(dts): print(f"{dt:8.5f} " + " ".join(f"{err[k][i]:24.2e}" for k in err))
orders = {k: np.log2(e[-2]/e[-1]) for k, e in err.items()}
print("estimated orders:", {k: round(float(o), 2) for k, o in orders.items()})
plt.figure(figsize=(5.5, 3.8))
for k, e in err.items(): plt.loglog(dts, e, "o-", label=k)
plt.xlabel(r"$\Delta t$"); plt.ylabel("rel. error at T = 0.5"); plt.grid(); plt.legend(); plt.show()

Both exponential methods show the predicted orders, and the exponential trapezoidal rule is the most accurate of the four. The remaining error is purely the quadrature error of the source term — the propagation of $u^n$ is exact. For sources that are polynomial in time of degree $\le 1$ the exponential trapezoidal rule is exact; smooth sources can be treated with higher-order exponential Runge–Kutta methods (Krogstad's or Hochbruck–Ostermann's fourth order schemes with $\varphi_3$), or by taking large steps with a Gauss quadrature of the integral, each node requiring one more $\varphi$-type evaluation.

Let us look at the solution itself (exponential trapezoidal rule, $\Delta t = 0.025$):

In [ ]:
dt = 0.025
E = KrylovExp(M, K, dt, freedofs, symmetric=False, tol=1e-8)
gfu.vec.data = u0vec
gfut = GridFunction(fes, multidim=0); gfut.AddMultiDimComponent(gfu.vec)
gnext = (invM*fvec(0.0)).Evaluate()
for k in range(int(round(2.0/dt))):
    gnow, gnext = gnext, (invM*fvec((k+1)*dt)).Evaluate()
    gfu.vec.data = E*gfu.vec + dt*E.Apply(gnow, lambda lam: phi(1, -dt*lam) - phi(2, -dt*lam)) + dt*E.Phi(2, gnext)
    if (k+1) % 4 == 0: gfut.AddMultiDimComponent(gfu.vec)
print("Krylov dimension of the last step:", E.info["m"])
Draw(gfut, mesh, interpolate_multidim=True, animate=True, min=-0.05, max=0.15);

### Remark: time-dependent Dirichlet data

Inhomogeneous, time-dependent boundary values $u = u_D(t)$ on $\Gamma_D$ are handled as in tutorial 3.1 (supplementary 3) by a lifting: write $u = \tilde u + \hat u_D(t)$ with an extension $\hat u_D(t)$ of the boundary data (e.g. `gfu.Set(uD, BND)`) and solve for $\tilde u$ with homogeneous data and the modified source $f - M\,\partial_t\hat u_D - K\hat u_D$. The time derivative of the extension is either known analytically or approximated by finite differences at the order of the time stepping method; the exponential integrator itself is unchanged.

In [ ]:
# tests
dt = 0.01; ex = sla.expm(-dt*Ad) @ u0np
E = KrylovExp(M, K, dt, freedofs, symmetric=False, tol=1e-11)
assert relerr((E*u0vec).Evaluate(), ex) < 1e-9 and E.info["m"] <= 30
assert 0.8 < orders["implicit Euler"] < 1.2 and 1.8 < orders["Crank-Nicolson"] < 2.2
assert 0.8 < orders["exponential Euler"] < 1.2 and 1.8 < orders["exponential trapezoidal"] < 2.3
assert err["exponential trapezoidal"][-1] < err["Crank-Nicolson"][-1]
print("all tests passed")

## Summary

* Non-symmetric operators: same code path, `symmetric=False` (Arnoldi, Hessenberg matrix, `umfpack`); convergence of shift-and-invert Krylov stays mesh-independent.
* Time-dependent sources are handled by exponential Runge–Kutta methods built from $\varphi_1, \varphi_2, \dots$; all are obtained from the same Krylov process by changing the scalar function.
* The exponential trapezoidal rule costs three Krylov runs per step (exp, $\varphi_1-\varphi_2$, $\varphi_2$) — or a single run on an augmented matrix, as done in `phipm`/`KIOPS` (see Part 2).

### References
* M. Hochbruck, A. Ostermann, *Explicit exponential Runge–Kutta methods for semilinear parabolic problems*, SIAM J. Numer. Anal. 43 (2005) 1069–1090. [doi:10.1137/040611434](https://doi.org/10.1137/040611434)
* M. Hochbruck, A. Ostermann, *Exponential integrators*, Acta Numerica 19 (2010) 209–286. [doi:10.1017/S0962492910000048](https://doi.org/10.1017/S0962492910000048)
* I. Moret, P. Novati, *RD-rational approximations of the matrix exponential*, BIT 44 (2004) 595–615 (shift-and-invert Arnoldi for non-symmetric matrices).
* M. Crouzeix, C. Palencia, *The numerical range is a $(1+\sqrt2)$-spectral set*, SIAM J. Matrix Anal. Appl. 38 (2017) 649–655. [doi:10.1137/17M1116672](https://doi.org/10.1137/17M1116672)
* J. Niesen, W. M. Wright, *Algorithm 919: A Krylov subspace algorithm for evaluating the φ-functions appearing in exponential integrators*, ACM Trans. Math. Softw. 38 (2012). [doi:10.1145/2168773.2168781](https://doi.org/10.1145/2168773.2168781)